In [ ]:
# ============================================================
# CELL 1 — Imports & settings
# Run once. Edit settings here before starting.
# ============================================================

import numpy as np
import open3d as o3d
import os, sys
import matplotlib.pyplot as plt
from scipy.ndimage import uniform_filter1d

# ── File & frame ──────────────────────────────────────────
INPUT_FILE  = r'C:\ROSBAGS VERWIJDER NA BEP\14_40_00\rosbag\rosbag_0.mcap'


# ── Sensor topics ─────────────────────────────────────────
LIDAR_TOPICS = [
    "/rslidar/M1P_deskewed",
    "/rslidar/helios_R",
    "/rslidar/helios_L",
]

# ── Sensor transforms (from Foxglove TF tree) ─────────────
SENSOR_TRANSFORMS = {
    "/rslidar/M1P_deskewed":  {"translation": [ 0.800,  0.000, 0.876], "rotation_rpy_deg": [0.1, -1.7,  1.7]},
    "/rslidar/helios_R":      {"translation": [-0.743, -0.243, 0.857], "rotation_rpy_deg": [2.4, -2.6, -179.9]},
    "/rslidar/helios_L":      {"translation": [-0.745,  0.191, 0.876], "rotation_rpy_deg": [3.5, -2.1,  179.9]},
}

# ── CSF parameters ────────────────────────────────────────
CLOTH_RESOLUTION = 0.5
SLOPE_SMOOTH     = True
THRESHOLD        = 0.4

# ── Profile parameters ────────────────────────────────────
Y_SLICE       = 0.5    # ±meters from centerline to include
BIN_SIZE      = 0.2    # meters per bin along X
SMOOTH_WINDOW = 10      # bins to smooth over

# ── Output ────────────────────────────────────────────────
OUTPUT_PLOT = r'C:\ROSBAGS VERWIJDER NA BEP\ground_profile.png'

print("Settings loaded.")


In [ ]:
# ============================================================
# CELL 1b — Frame estimator from timestamp
# Reads actual timestamps from the mcap file automatically.
# Run once after Cell 1, then rerun whenever you change timestamp.
# ============================================================

from mcap.reader import make_reader
from mcap_ros2.decoder import DecoderFactory

TARGET_TIMESTAMP = 1777639310.322213694  # <-- paste your Foxglove timestamp here

REFERENCE_TOPIC = "/rslidar/M1P_deskewed"  # topic to use for frame counting

print(f"Scanning timestamps from '{REFERENCE_TOPIC}'...")
print("(This may take a moment...)")

timestamps = []
with open(INPUT_FILE, "rb") as f:
    reader = make_reader(f, decoder_factories=[DecoderFactory()])
    for schema, channel, message, ros_msg in reader.iter_decoded_messages(topics=[REFERENCE_TOPIC]):
        # log_time is in nanoseconds — convert to seconds
        timestamps.append(message.log_time / 1e9)

timestamps = np.array(timestamps)
print(f"Found {len(timestamps)} frames")
print(f"Recording start: {timestamps[0]:.3f}")
print(f"Recording end:   {timestamps[-1]:.3f}")
print(f"Duration:        {timestamps[-1] - timestamps[0]:.2f}s")

# Find closest frame to target timestamp
diffs          = np.abs(timestamps - TARGET_TIMESTAMP)
frame_estimate = int(np.argmin(diffs))
closest_time   = timestamps[frame_estimate]
time_error     = abs(closest_time - TARGET_TIMESTAMP)

print(f"\nTarget timestamp:  {TARGET_TIMESTAMP:.3f}")
print(f"Closest timestamp: {closest_time:.3f}  (error: {time_error*1000:.1f}ms)")
print(f"→ Estimated frame: {frame_estimate}")

# Auto-update FRAME_INDEX
FRAME_INDEX = frame_estimate
print(f"\nFRAME_INDEX updated to {FRAME_INDEX} — rerun from Cell 3")

In [ ]:
# ============================================================
# CELL 2 — Helper functions
# Run once.
# ============================================================

def _ros_pc2_to_numpy(msg):
    import numpy as np
    field_map = {f.name: f for f in msg.fields}
    if not all(k in field_map for k in ("x", "y", "z")):
        return None

    # Read the raw bytes as a structured numpy array — no Python loop needed
    dtype_fields = []
    for f in sorted(msg.fields, key=lambda f: f.offset):
        # map ROS field datatypes to numpy
        ros_to_np = {1: 'i1', 2: 'u1', 3: 'i2', 4: 'u2', 5: 'i4', 6: 'u4', 7: 'f4', 8: 'f8'}
        np_type = ros_to_np.get(f.datatype, 'f4')
        dtype_fields.append((f.name, np_type))

    dtype   = np.dtype(dtype_fields)
    n       = msg.width * msg.height
    raw     = np.frombuffer(bytes(msg.data), dtype=np.uint8)
    
    # Extract x, y, z using offsets directly — handles non-packed structs
    x_off = field_map["x"].offset
    y_off = field_map["y"].offset
    z_off = field_map["z"].offset
    step  = msg.point_step

    # Reshape into (n_points, point_step) and slice out x y z
    raw_2d = raw[:n * step].reshape(n, step)
    x = raw_2d[:, x_off:x_off+4].view(np.float32).reshape(-1)
    y = raw_2d[:, y_off:y_off+4].view(np.float32).reshape(-1)
    z = raw_2d[:, z_off:z_off+4].view(np.float32).reshape(-1)

    xyz = np.stack([x, y, z], axis=1)
    return xyz[np.isfinite(xyz).all(axis=1)]

In [ ]:
# ============================================================
# CELL 3 — Load raw point clouds from mcap
# SLOW — only rerun if you change FRAME_INDEX or INPUT_FILE
# ============================================================

print(f"Loading frame {FRAME_INDEX} from {len(LIDAR_TOPICS)} sensors...")
raw_clouds = {}
for topic in LIDAR_TOPICS:
    pts = load_mcap_topic(INPUT_FILE, topic, FRAME_INDEX)
    if pts is not None:
        raw_clouds[topic] = pts

print(f"\nLoaded {len(raw_clouds)} sensors.")


In [ ]:
# ============================================================
# CELL 4 — Apply sensor transforms & merge into one cloud
# Rerun if you change SENSOR_TRANSFORMS
# ============================================================

transformed = {}
for topic, pts in raw_clouds.items():
    tf  = SENSOR_TRANSFORMS[topic]
    transformed[topic] = transform_points(pts, tf["translation"], tf["rotation_rpy_deg"])

merged = np.vstack(list(transformed.values()))
print(f"Merged cloud: {len(merged):,} points")
print(f"  X  {merged[:,0].min():.2f} → {merged[:,0].max():.2f} m")
print(f"  Y  {merged[:,1].min():.2f} → {merged[:,1].max():.2f} m")
print(f"  Z  {merged[:,2].min():.2f} → {merged[:,2].max():.2f} m")


In [ ]:
# ============================================================
# CELL 5 — CSF ground filter
# Rerun if you change CLOTH_RESOLUTION, SLOPE_SMOOTH, THRESHOLD
# ============================================================

import CSF

csf = CSF.CSF()
csf.params.bSloopSmooth     = SLOPE_SMOOTH
csf.params.cloth_resolution = CLOTH_RESOLUTION
csf.params.threshold        = THRESHOLD
csf.setPointCloud(merged)

ground_idx     = CSF.VecInt()
non_ground_idx = CSF.VecInt()
csf.do_filtering(ground_idx, non_ground_idx)

ground     = merged[np.array(list(ground_idx))]
non_ground = merged[np.array(list(non_ground_idx))]

pct_g  = 100 * len(ground) / len(merged)
pct_ng = 100 * len(non_ground) / len(merged)
print(f"Ground points:     {len(ground):,}  ({pct_g:.1f}%)")
print(f"Non-ground points: {len(non_ground):,}  ({pct_ng:.1f}%)")


In [ ]:
# ============================================================
# CELL 6 — Slice along centerline & bin by X
# Rerun if you change Y_SLICE or BIN_SIZE
# ============================================================

# Slice to centerline strip
mask      = np.abs(ground[:, 1]) <= Y_SLICE
slice_pts = ground[mask]
print(f"Points within Y ±{Y_SLICE}m: {len(slice_pts):,}")

if len(slice_pts) < 10:
    print("ERROR: Too few points. Try increasing Y_SLICE.")
else:
    x = -slice_pts[:, 0]
    z = slice_pts[:, 2]

    # Bin by X
    x_min = np.floor(x.min() / BIN_SIZE) * BIN_SIZE
    x_max = np.ceil(x.max()  / BIN_SIZE) * BIN_SIZE
    bins  = np.arange(x_min, x_max + BIN_SIZE, BIN_SIZE)

    bin_centers, bin_heights = [], []
    for i in range(len(bins) - 1):
        in_bin = (x >= bins[i]) & (x < bins[i+1])
        if in_bin.sum() >= 3:
            bin_centers.append((bins[i] + bins[i+1]) / 2)
            bin_heights.append(np.median(z[in_bin]))

    bin_centers = np.array(bin_centers)
    bin_heights = np.array(bin_heights)
    print(f"Bins with data:   {len(bin_centers)}")
    print(f"X range:          {bin_centers.min():.1f} → {bin_centers.max():.1f} m")


In [ ]:
# ============================================================
# CELL 7 — Compute slope & curvature
# Rerun if you change SMOOTH_WINDOW
# ============================================================

z_smooth         = uniform_filter1d(bin_heights, size=SMOOTH_WINDOW)
dz               = np.gradient(z_smooth, bin_centers)
slope_deg        = np.degrees(np.arctan(dz))
d2z              = np.gradient(dz, bin_centers)
curvature        = d2z
slope_smooth     = uniform_filter1d(slope_deg, size=SMOOTH_WINDOW)
curvature_smooth = uniform_filter1d(curvature, size=SMOOTH_WINDOW)

print(f"Slope range:     {slope_smooth.min():.2f}° → {slope_smooth.max():.2f}°")
print(f"Curvature range: {curvature_smooth.min():.4f} → {curvature_smooth.max():.4f} m⁻¹")


In [ ]:
# ============================================================
# CELL 8 — Plot ground height, slope & curvature
# Rerun freely to tweak the graph appearance
# ============================================================
import numpy as np

# Mask invalid values so lines break at gaps instead of interpolating
z_smooth_masked         = np.ma.masked_invalid(z_smooth)
bin_heights_masked      = np.ma.masked_invalid(bin_heights)
slope_smooth_masked     = np.ma.masked_invalid(slope_smooth)
slope_deg_masked        = np.ma.masked_invalid(slope_deg)
curvature_smooth_masked = np.ma.masked_invalid(curvature_smooth)
curvature_masked        = np.ma.masked_invalid(curvature)

fig, axes = plt.subplots(3, 1, figsize=(14, 10), sharex=True)
fig.suptitle(f"Ground Profile — Frame {FRAME_INDEX}  |  Y slice ±{Y_SLICE}m",
             fontsize=14, fontweight='bold')

# ── Height ────────────────────────────────────────────────
ax1 = axes[0]
ax1.plot(bin_centers, bin_heights_masked, color='lightsteelblue', linewidth=1.0, alpha=0.7, label='Raw height')
ax1.plot(bin_centers, z_smooth_masked,    color='royalblue',      linewidth=2.0, label='Smoothed height')
ax1.set_ylabel("Height Z (m)", fontsize=11)
ax1.set_title("Ground Height", fontsize=11)
ax1.legend(fontsize=9)
ax1.grid(True, alpha=0.3)
ax1.axhline(0, color='gray', linewidth=0.5, linestyle='--')

# ── Slope ─────────────────────────────────────────────────
ax2 = axes[1]
ax2.plot(bin_centers, slope_deg_masked,    color='moccasin',   linewidth=1.0, alpha=0.7, label='Raw slope')
ax2.plot(bin_centers, slope_smooth_masked, color='darkorange', linewidth=2.0, label='Smoothed slope')
ax2.fill_between(bin_centers, slope_smooth_masked, 0,
                 where=(slope_smooth_masked >= 0), alpha=0.15, color='red',  label='Uphill')
ax2.fill_between(bin_centers, slope_smooth_masked, 0,
                 where=(slope_smooth_masked <  0), alpha=0.15, color='blue', label='Downhill')
ax2.set_ylabel("Slope (degrees)", fontsize=11)
ax2.set_title("Ground Slope along Forward Direction", fontsize=11)
ax2.legend(fontsize=9)
ax2.grid(True, alpha=0.3)
ax2.axhline(0, color='gray', linewidth=0.5, linestyle='--')

# ── Curvature ─────────────────────────────────────────────
ax3 = axes[2]
ax3.plot(bin_centers, curvature_masked,        color='lightgreen', linewidth=1.0, alpha=0.7, label='Raw curvature')
ax3.plot(bin_centers, curvature_smooth_masked, color='green',      linewidth=2.0, label='Smoothed curvature')
ax3.fill_between(bin_centers, curvature_smooth_masked, 0,
                 where=(curvature_smooth_masked >= 0), alpha=0.15, color='orange', label='Convex (crest)')
ax3.fill_between(bin_centers, curvature_smooth_masked, 0,
                 where=(curvature_smooth_masked <  0), alpha=0.15, color='purple', label='Concave (dip)')
ax3.set_ylabel("Curvature (m⁻¹)", fontsize=11)
ax3.set_xlabel("Forward distance X (m)", fontsize=11)
ax3.set_title("Vertical Curvature along Forward Direction", fontsize=11)
ax3.legend(fontsize=9)
ax3.grid(True, alpha=0.3)
ax3.axhline(0, color='gray', linewidth=0.5, linestyle='--')

for ax in axes:
    ax.set_xlim(-10, 20)

plt.tight_layout()
plt.savefig(OUTPUT_PLOT, dpi=150, bbox_inches='tight')
print(f"Saved to: {OUTPUT_PLOT}")
plt.show()

In [ ]:
from mcap.reader import make_reader

count = 0
with open(INPUT_FILE, "rb") as f:
    reader = make_reader(f)
    for schema, channel, message in reader.iter_messages(
            topics=["/odometry/local"]):
        print(f"t={message.log_time/1e9:.3f}  channel={channel.topic}")
        count += 1
        if count >= 3:
            break

print(f"Total messages read: {count}")